In [1]:
# =========================
# Cell 1. Install
# =========================
!pip -q uninstall -y transformers tokenizers accelerate datasets peft bitsandbytes trl llmcompressor compressed-tensors

!pip -q install "transformers>=4.57.0"
!pip -q install "tokenizers>=0.22.0"
!pip -q install "accelerate>=1.10.0"
!pip -q install "datasets>=4.4.0"
!pip -q install "peft>=0.17.0"
!pip -q install "bitsandbytes>=0.47.0"
!pip -q install "trl>=0.29.0"
!pip -q install "llmcompressor"
!pip -q install "compressed-tensors>=0.13.0"

print("설치 완료. 가능하면 런타임 재시작 후 다음 셀부터 실행.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 122.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 127.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchtune 0.6.1 requires datasets, which is not installed.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.5/527.5 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 54.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 43.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 14.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29

In [2]:
# =========================
# Cell 2. Imports
# =========================
import os
import gc
import re
import math
import json
import torch
import shutil
from pathlib import Path
from difflib import SequenceMatcher

from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, PeftModel
from trl import GRPOConfig, GRPOTrainer
from llmcompressor import oneshot
from llmcompressor.modifiers.quantization import GPTQModifier

os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

torch: 2.10.0+cu128
cuda: True
gpu: NVIDIA L4


In [3]:
# =========================
# Cell 3. Settings
# =========================
MODEL_ID = "LGAI-EXAONE/EXAONE-4.0-1.2B"

GRPO_ADAPTER_DIR = "./grpo_adapter"
MERGED_MODEL_DIR = "./grpo_merged_model"
GPTQ_OUT_DIR = "./model"

DATASET_ID = "LGAI-EXAONE/MANTA-1M"
DATASET_SPLIT = "train"

MAX_TRAIN_SAMPLES = 2000
MAX_PROMPT_TOKENS = 384
MAX_COMPLETION_TOKENS = 128

USE_4BIT_FOR_GRPO = True

# GRPO는 num_generations이 유효 batch size와 나누어떨어져야 함
PER_DEVICE_BATCH_SIZE = 1
GRAD_ACCUM = 4
NUM_GENERATIONS = 4

LEARNING_RATE = 5e-6
MAX_STEPS = 120
SAVE_STEPS = 40
LOGGING_STEPS = 5

# GPTQ
NUM_CALIBRATION_SAMPLES = 256
GPTQ_MAX_SEQUENCE_LENGTH = 512
SCHEME = "W4A16"
TARGETS = ["Linear"]
IGNORE = ["embed_tokens", "lm_head"]

In [4]:
# =========================
# Cell 4. Tokenizer Load
# =========================
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
)

if tokenizer.pad_token is None and tokenizer.eos_token is not None:
    tokenizer.pad_token = tokenizer.eos_token

print("pad_token:", tokenizer.pad_token)
print("eos_token:", tokenizer.eos_token)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

pad_token: [PAD]
eos_token: [|endofturn|]


In [5]:
# =========================
# Cell 5. Build GRPO Dataset
# =========================
raw_ds = load_dataset(
    DATASET_ID,
    split=f"{DATASET_SPLIT}[:{MAX_TRAIN_SAMPLES}]",
)

def build_example(example):
    convs = example["conversations"]

    if len(convs) < 2:
        return None

    if convs[-1]["role"] != "assistant":
        return None

    answer = convs[-1]["content"].strip()
    prompt_messages = convs[:-1]

    try:
        prompt_text = tokenizer.apply_chat_template(
            prompt_messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    except Exception:
        return None

    tokenized = tokenizer(
        prompt_text,
        truncation=True,
        max_length=MAX_PROMPT_TOKENS,
        return_attention_mask=False,
    )

    prompt_text = tokenizer.decode(tokenized["input_ids"], skip_special_tokens=False)

    if len(answer.strip()) == 0:
        return None

    return {
        "prompt": prompt_text,
        "answer": answer,
    }

rows = []
for ex in raw_ds:
    built = build_example(ex)
    if built is not None:
        rows.append(built)

train_ds = Dataset.from_list(rows)

print("GRPO train rows:", len(train_ds))
print(train_ds[0]["prompt"][:500])
print("-----")
print(train_ds[0]["answer"][:300])

README.md: 0.00B [00:00, ?B/s]

data/train.parquet:   0%|          | 0.00/1.94G [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000000 [00:00<?, ? examples/s]

GRPO train rows: 2000
[|user|]
In examining the impact of open-access publishing on the dissemination of mathematical research, a common argument posits that it significantly enhances accessibility and collaboration. However, flaw this reasoning by discussing potential drawbacks that could limit its effectiveness in ensuring equitable access to high-quality mathematical knowledge across diverse global communities.[|endofturn|]
[|assistant|]
<think>

</think>


-----
To critically examine the argument that open-access publishing significantly enhances accessibility and collaboration in mathematical research, while also addressing potential drawbacks that could limit its effectiveness in ensuring equitable access, we can break down the analysis into several key p


In [6]:
# =========================
# Cell 6. Quick Check
# =========================
lengths = []
for i in range(min(100, len(train_ds))):
    lengths.append(len(tokenizer(train_ds[i]["prompt"])["input_ids"]))

print("sample count:", len(train_ds))
print("avg prompt tokens (first 100):", sum(lengths) / len(lengths))
print("max prompt tokens (first 100):", max(lengths))

sample count: 2000
avg prompt tokens (first 100): 114.7
max prompt tokens (first 100): 384


In [7]:
# =========================
# Cell 7. Reward Functions
# =========================
def _normalize_text(text):
    text = text.strip().lower()
    text = re.sub(r"\s+", " ", text)
    return text

def _similarity(a, b):
    a = _normalize_text(a)
    b = _normalize_text(b)
    if len(a) == 0 and len(b) == 0:
        return 1.0
    return SequenceMatcher(None, a, b).ratio()

def answer_similarity_reward(prompts, completions, answer, **kwargs):
    rewards = []
    for comp, gold in zip(completions, answer):
        score = _similarity(comp, gold)
        rewards.append(float(score * 2.0))
    return rewards

def length_penalty_reward(prompts, completions, answer, **kwargs):
    rewards = []
    for comp in completions:
        n = len(comp.strip())
        if n < 8:
            rewards.append(-1.0)
        elif n > 600:
            rewards.append(-0.5)
        else:
            rewards.append(0.2)
    return rewards

def contains_korean_reward(prompts, completions, answer, **kwargs):
    rewards = []
    for comp in completions:
        if re.search(r"[가-힣]", comp):
            rewards.append(0.2)
        else:
            rewards.append(0.0)
    return rewards

In [14]:
# =========================
# Cell 8. Load 4bit Base Model for GRPO (BF16 for L4)
# =========================
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

grpo_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

grpo_model.config.use_cache = False
grpo_model.gradient_checkpointing_enable()

print("GRPO base model loaded")

# dtype 확인
dtypes = {}
for n, p in grpo_model.named_parameters():
    dtypes[str(p.dtype)] = dtypes.get(str(p.dtype), 0) + 1
print("parameter dtype summary:", dtypes)

GRPO base model loaded
parameter dtype summary: {'torch.bfloat16': 122, 'torch.uint8': 210}


In [15]:
# =========================
# Cell 9. LoRA Config
# =========================
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

In [16]:
# =========================
# Cell 10. GRPO Config (BF16 for L4)
# =========================
grpo_args = GRPOConfig(
    output_dir=GRPO_ADAPTER_DIR,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    max_steps=MAX_STEPS,
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    save_strategy="steps",
    save_total_limit=2,

    fp16=False,
    bf16=True,

    report_to="none",
    remove_unused_columns=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},

    num_generations=NUM_GENERATIONS,
    max_completion_length=MAX_COMPLETION_TOKENS,
    temperature=0.9,
    top_p=0.95,
    beta=0.0,
    loss_type="dapo",
    use_vllm=False,
)

In [17]:
# =========================
# Cell 10.5 Force trainable params to FP16
# =========================
def print_trainable_dtypes(model):
    counts = {}
    trainable = 0
    total = 0
    for n, p in model.named_parameters():
        total += p.numel()
        if p.requires_grad:
            trainable += p.numel()
            counts[str(p.dtype)] = counts.get(str(p.dtype), 0) + 1
    print("trainable params:", trainable)
    print("total params:", total)
    print("trainable dtype summary:", counts)

print("[Before PEFT attach / training]")
print_trainable_dtypes(grpo_model)

[Before PEFT attach / training]
trainable params: 209843968
total params: 744617728
trainable dtype summary: {'torch.bfloat16': 122}


In [18]:
# =========================
# Cell 11. Build GRPOTrainer
# =========================
trainer = GRPOTrainer(
    model=grpo_model,
    args=grpo_args,
    reward_funcs=[
        answer_similarity_reward,
        length_penalty_reward,
        contains_korean_reward,
    ],
    train_dataset=train_ds,
    processing_class=tokenizer,
    peft_config=peft_config,
)

# trainable 파라미터 dtype 확인
counts = {}
for name, param in trainer.model.named_parameters():
    if param.requires_grad:
        counts[str(param.dtype)] = counts.get(str(param.dtype), 0) + 1
print("trainable dtype summary:", counts)

print("GRPOTrainer ready")

trainable dtype summary: {'torch.bfloat16': 420}
GRPOTrainer ready


In [19]:
# =========================
# Cell 12. Train GRPO
# =========================
trainer.train()

Could not estimate the number of tokens of the input, floating-point operations will not be computed


Step,Training Loss
5,0.000000
10,0.015500
15,-0.000000
20,-0.000000
25,-0.000000
30,0.000000
35,-0.000000
40,0.000000
45,0.000000
50,-0.036500


TrainOutput(global_step=120, training_loss=-0.0012504237684576461, metrics={'train_runtime': 2630.4814, 'train_samples_per_second': 0.182, 'train_steps_per_second': 0.046, 'total_flos': 0.0, 'train_loss': -0.0012504237684576461})

In [20]:
# =========================
# Cell 13. Save Adapter
# =========================
trainer.model.save_pretrained(GRPO_ADAPTER_DIR)
tokenizer.save_pretrained(GRPO_ADAPTER_DIR)

print("adapter saved:", GRPO_ADAPTER_DIR)

adapter saved: ./grpo_adapter


In [21]:
# =========================
# Cell 14. Cleanup Before Merge
# =========================
del trainer
del grpo_model
gc.collect()
torch.cuda.empty_cache()

print("cleanup done")

cleanup done


In [22]:
# =========================
# Cell 15. Merge LoRA into FP16 Base
# =========================
base_model_fp16 = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    torch_dtype=torch.float16,
    device_map="auto",
)

merged_model = PeftModel.from_pretrained(
    base_model_fp16,
    GRPO_ADAPTER_DIR,
)

merged_model = merged_model.merge_and_unload()

os.makedirs(MERGED_MODEL_DIR, exist_ok=True)
merged_model.save_pretrained(MERGED_MODEL_DIR)
tokenizer.save_pretrained(MERGED_MODEL_DIR)

print("merged model saved:", MERGED_MODEL_DIR)

merged model saved: ./grpo_merged_model


In [23]:
# =========================
# Cell 16. Quick Generation Test
# =========================
test_prompt = "사용자: 자기소개를 3문장으로 해줘.\n어시스턴트:"

inputs = tokenizer(test_prompt, return_tensors="pt").to(merged_model.device)

with torch.no_grad():
    outputs = merged_model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=True,
        temperature=0.8,
        top_p=0.95,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

사용자: 자기소개를 3문장으로 해줘.
어시스턴트: 어시스턴트 = 원래 이름 + 원래 이름 + 원래 이름 + 원래 이름 + 원래 이름 + 원래 이름 + 원래 이름 + 원래 이름 + 원래 이름 + 원래 이름 + 원래 이름 + 원래 이름 + 원래 이름 + 원래 이름 + 원래 이름 + 원래 성 + 원래 성 + 원래 성 + 원래 성 + 원래 성 + 원래 성 + 원래 성 + 원래 성 + 원래 성 + 원래 성 + 원래


In [24]:
# =========================
# Cell 17. Load Merged Model for GPTQ
# =========================
del merged_model
del base_model_fp16
gc.collect()
torch.cuda.empty_cache()

quant_tokenizer = AutoTokenizer.from_pretrained(
    MERGED_MODEL_DIR,
    trust_remote_code=True,
)

if quant_tokenizer.pad_token is None and quant_tokenizer.eos_token is not None:
    quant_tokenizer.pad_token = quant_tokenizer.eos_token

quant_model = AutoModelForCausalLM.from_pretrained(
    MERGED_MODEL_DIR,
    trust_remote_code=True,
    torch_dtype=torch.float16,
    device_map="auto",
)

quant_model.eval()

print("merged model reloaded for GPTQ")

The tokenizer you are loading from './grpo_merged_model' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


merged model reloaded for GPTQ


In [25]:
# =========================
# Cell 18. GPTQ Calibration Dataset
# =========================
calib_raw = load_dataset(
    DATASET_ID,
    split=f"{DATASET_SPLIT}[:{NUM_CALIBRATION_SAMPLES}]",
)

def preprocess_calib(example):
    return {
        "text": quant_tokenizer.apply_chat_template(
            example["conversations"],
            add_generation_prompt=True,
            tokenize=False,
        )
    }

calib_ds = calib_raw.map(preprocess_calib)
print(calib_ds[0]["text"][:500])

Map:   0%|          | 0/256 [00:00<?, ? examples/s]

[|user|]
In examining the impact of open-access publishing on the dissemination of mathematical research, a common argument posits that it significantly enhances accessibility and collaboration. However, flaw this reasoning by discussing potential drawbacks that could limit its effectiveness in ensuring equitable access to high-quality mathematical knowledge across diverse global communities.[|endofturn|]
[|assistant|]
<think>

</think>

To critically examine the argument that open-access publis


In [26]:
# =========================
# Cell 19. GPTQ Recipe
# =========================
recipe = [
    GPTQModifier(
        scheme=SCHEME,
        targets=TARGETS,
        ignore=IGNORE,
    )
]

print(recipe)

[GPTQModifier(config_groups=None, targets=['Linear'], ignore=['embed_tokens', 'lm_head'], scheme='W4A16', kv_cache_scheme=None, weight_observer=None, input_observer=None, output_observer=None, observer=None, bypass_divisibility_checks=False, index=None, group=None, start=None, end=None, update=None, initialized_=False, finalized_=False, started_=False, ended_=False, sequential_targets=None, block_size=128, dampening_frac=0.01, actorder=static, offload_hessians=False)]


In [27]:
# =========================
# Cell 21. Save Quantized Model
# =========================
os.makedirs(GPTQ_OUT_DIR, exist_ok=True)

quant_model.save_pretrained(GPTQ_OUT_DIR, save_compressed=True)
quant_tokenizer.save_pretrained(GPTQ_OUT_DIR)

print("quantized model saved:", GPTQ_OUT_DIR)

quantized model saved: ./model


In [28]:
# =========================
# Cell 22. Zip Submission
# =========================
zip_name = "0.619_GRPO"

shutil.make_archive(
    base_name=zip_name,
    format="zip",
    root_dir=".",
    base_dir=GPTQ_OUT_DIR,
)

print(f"{zip_name}.zip 생성 완료")

0.619_GRPO.zip 생성 완료


In [29]:
# =========================
# Cell 23. Check Zip Size
# =========================
size_mb = os.path.getsize(f"{zip_name}.zip") / (1024 * 1024)
print(f"zip size: {size_mb:.2f} MB")

zip size: 2087.68 MB


In [1]:
!pip -q install lm-eval[vllm]

In [2]:
!python -m lm_eval --model vllm \
  --model_args pretrained=/content/model,gpu_memory_utilization=0.85,max_gen_toks=2048 \
  --tasks gsm8k \
  --batch_size auto \
  --apply_chat_template \
  --limit 512

2026-03-14:14:06:01 WARNING  [config.evaluate_config:281] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-03-14:14:06:01 INFO     [config.evaluate_config:301] Using default fewshot_as_multiturn=True.
2026-03-14:14:06:07 INFO     [_cli.run:376] Selected Tasks: ['gsm8k']
2026-03-14:14:06:08 INFO     [evaluator:211] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-03-14:14:06:08 INFO     [evaluator:236] Initializing vllm model, with arguments: {'pretrained': '/content/model', 'gpu_memory_utilization': 0.85, 'max_gen_toks': 2048}
2026-03-14 14:06:15.706964: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-14 14:06:15.726163: E external/loca

In [3]:
!pip -q install vllm==0.14.1

In [4]:
!git clone -q https://github.com/vllm-project/vllm.git

fatal: destination path 'vllm' already exists and is not an empty directory.


In [5]:
!vllm bench throughput \
  --model /content/model \
  --dataset-name random \
  --num-prompts 100 \
  --random-input-len 100 \
  --random-output-len 200 \
  --random-range-ratio 0.0 \
  --seed 42 \
  --gpu-memory-utilization 0.85

2026-03-14 14:13:37.637669: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-14 14:13:37.656485: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773497617.680496   21011 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773497617.687604   21011 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773497617.704443   21011 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 